# 第16章　学習の高速化と大規模化 ― 速く、大きく回す

**『医療診断支援AIを自分で作る（基礎編）』のコード**

本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。

- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。
- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。
- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**

リポジトリ: https://github.com/kewel-corp/book-basic

## 16.1　まず、無駄をなくす ― データ供給の最適化

In [ ]:
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True,
                          num_workers=8,        # 並列でデータを読み込む
                          pin_memory=True,       # GPU転送を高速化
                          persistent_workers=True)  # ワーカーを使い回す

## GPUが遊んでいないか、を確かめる ― worked example

```text
| GPU-Util |   Memory-Usage      |
|   14%    | 9200MiB / 24576MiB |   ← 使用率が低い＝GPUが手待ち＝データ供給が律速
```

## 16.2　混合精度で、倍速・省メモリ

In [ ]:
from torch.amp import autocast, GradScaler
scaler = GradScaler("cuda")
for x, y in loader:
    optimizer.zero_grad()
    with autocast("cuda"):              # FP16/FP32を自動で振り分け
        loss = criterion(model(x), y)
    scaler.scale(loss).backward()
    scaler.step(optimizer); scaler.update()

## 16.3　勾配累積で、大きなバッチを再現する

In [ ]:
accum = 4                                # 4回ぶんためて1回更新
n_batches = len(loader)
optimizer.zero_grad()                    # ループ前に一度リセット（前エポックの余りを持ち込まない）
for i, (x, y) in enumerate(loader):
    # この更新でまとめる本数。末尾の端数グループでは4本に満たないので、実数で割る
    group = min(accum, n_batches - (i // accum) * accum)
    with autocast("cuda"):
        loss = criterion(model(x), y) / group
    scaler.scale(loss).backward()
    if (i + 1) % accum == 0 or (i + 1) == n_batches:   # 端数グループでも必ず更新する
        scaler.step(optimizer); scaler.update(); optimizer.zero_grad()

## マルチGPU（DDP）とAMPを、数字で確かめる

In [ ]:

# torchrun --nproc_per_node=4 train_ddp.py で起動（GPU 4枚）
import torch, os
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data.distributed import DistributedSampler

dist.init_process_group("nccl")
local_rank  = int(os.environ["LOCAL_RANK"])      # ノード内でのGPU番号。デバイス指定用
torch.cuda.set_device(local_rank)
global_rank = dist.get_rank()                    # 全プロセス通しの番号。保存担当の判定用
model = DDP(build_model().cuda(local_rank), device_ids=[local_rank])

sampler = DistributedSampler(train_ds)                  # 各GPUへ分配（既定では件数調整のため重複を含むことがある）
loader  = DataLoader(train_ds, batch_size=8, sampler=sampler, pin_memory=True)

for epoch in range(epochs):
    sampler.set_epoch(epoch)                            # ←これを忘れると毎エポック同じ並び
    for x, y in loader:
        ...                                             # 学習ループは通常どおり
if global_rank == 0:                                    # 保存は全体で1プロセスだけ
    torch.save(model.module.state_dict(), "best.pth")   # LOCAL_RANKで判定すると、複数マシンでは
                                                        # 各ノードの0番が同じファイルへ同時に書く

## ColabのGPUでAMPを効かせる ― まず種類を確かめ、数字で測る

In [ ]:
import torch
assert torch.cuda.is_available(), "ランタイム→ランタイムのタイプを変更→GPU を選ぶ"
print(torch.cuda.get_device_name(0))         # 例: Tesla T4 / NVIDIA L4 / NVIDIA A100
cap = torch.cuda.get_device_capability(0)     # (7,5)=T4  (8,9)=L4  (8,0)=A100
print("compute capability", cap, "→ bfloat16が使える:", cap[0] >= 8)

In [ ]:
def amp_context():
    cap = torch.cuda.get_device_capability(0)
    if cap[0] >= 8:                                   # A100 / L4 など
        return torch.bfloat16, False                 # bf16、スケーラ不要
    return torch.float16, True                        # T4 など、fp16 + スケーラ必須

In [ ]:
import time
from torch.amp import autocast, GradScaler

def bench(build_model, use_amp, steps=50):
    amp_dtype, need_scaler = amp_context()
    model = build_model().cuda()
    opt = torch.optim.AdamW(model.parameters(), lr=1e-4)
    scaler = GradScaler("cuda", enabled=(use_amp and need_scaler))
    crit = torch.nn.CrossEntropyLoss()
    x = torch.randn(16, 3, 224, 224, device="cuda")   # ダミー入力（純粋に計算だけ測る）
    y = torch.randint(0, 2, (16,), device="cuda")
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize(); t0 = time.time()        # GPUは非同期。計測前後で同期する
    for _ in range(steps):
        opt.zero_grad()
        with autocast("cuda", dtype=amp_dtype, enabled=use_amp):
            loss = crit(model(x), y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
    torch.cuda.synchronize()
    ms = (time.time() - t0) / steps * 1000
    gb = torch.cuda.max_memory_allocated() / 1e9
    return ms, gb

import timm
build = lambda: timm.create_model("efficientnet_b0", num_classes=2)
print("FP32:", bench(build, use_amp=False))
print("AMP :", bench(build, use_amp=True))